Aim

To implement a Sequence-to-Sequence (Seq2Seq) model using LSTM with real text sequence data in TensorFlow and Keras.

In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    GRU,
    Dense
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# Text Dataset
questions = [
    "what is your name",
    "how are you",
    "good evening",
    "where are you",
    "thank you"
]

answers = [
    "my name is bot",
    "i am fine",
    "good evening friend",
    "i am here",
    "welcome"
]

# Tokenizer Initialization
text_tokenizer = Tokenizer()

text_tokenizer.fit_on_texts(
    questions + answers
)

# Convert Sentences to Sequences
question_sequences = text_tokenizer.texts_to_sequences(
    questions
)

answer_sequences = text_tokenizer.texts_to_sequences(
    answers
)

# Maximum Sequence Length
sequence_size = max(
    max(len(item) for item in question_sequences),
    max(len(item) for item in answer_sequences)
)

# Padding Sequences
encoder_data = pad_sequences(
    question_sequences,
    maxlen=sequence_size,
    padding='post'
)

decoder_data = pad_sequences(
    answer_sequences,
    maxlen=sequence_size,
    padding='post'
)

# Vocabulary Size
total_words = len(text_tokenizer.word_index) + 1

# Convert Decoder Output to One-Hot Encoding
decoder_targets = to_categorical(
    decoder_data,
    num_classes=total_words
)

# Hidden Units
hidden_units = 128

# Encoder Input
encoder_input_layer = Input(
    shape=(sequence_size,)
)

# Encoder Embedding
encoder_embedding_layer = Embedding(
    input_dim=total_words,
    output_dim=hidden_units
)(encoder_input_layer)

# Encoder GRU
encoder_gru = GRU(
    hidden_units,
    return_state=True
)

encoder_output, encoder_state = encoder_gru(
    encoder_embedding_layer
)

# Decoder Input
decoder_input_layer = Input(
    shape=(sequence_size,)
)

# Decoder Embedding
decoder_embedding_layer = Embedding(
    input_dim=total_words,
    output_dim=hidden_units
)(decoder_input_layer)

# Decoder GRU
decoder_gru = GRU(
    hidden_units,
    return_sequences=True,
    return_state=True
)

decoder_output, _ = decoder_gru(
    decoder_embedding_layer,
    initial_state=encoder_state
)

# Dense Output Layer
output_layer = Dense(
    total_words,
    activation='softmax'
)

final_output = output_layer(decoder_output)

# Build Seq2Seq Model
seq2seq_network = Model(
    [encoder_input_layer, decoder_input_layer],
    final_output
)

# Display Summary
seq2seq_network.summary()

# Compile Model
seq2seq_network.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Train Seq2Seq Model
training_process = seq2seq_network.fit(
    [encoder_data, decoder_data],
    decoder_targets,
    epochs=150,
    batch_size=1,
    verbose=1
)

# Evaluate Model
model_loss, model_accuracy = seq2seq_network.evaluate(
    [encoder_data, decoder_data],
    decoder_targets,
    verbose=0
)

print("\nSequence Model Loss :", model_loss)
print("Sequence Model Accuracy :", model_accuracy)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 4, 128)    │      2,560 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 4, 128)    │      2,560 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ [(None, 128),     │     99,072 │ embedding[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ [(None, 4, 128),  │     99,072 │ embedding_1[0][0… │
│                     │ (None, 128)]      │            │ gru[0][1]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 4, 20)     │      2,580 │ gru_1[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 205,844 (804.08 KB)

 Trainable params: 205,844 (804.08 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.3000 - loss: 2.9545
Epoch 2/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3500 - loss: 2.7863 
Epoch 3/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3000 - loss: 2.5807
Epoch 4/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3000 - loss: 2.3042
Epoch 5/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.3000 - loss: 2.0776    
Epoch 6/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.4000 - loss: 1.9584
Epoch 7/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5500 - loss: 1.8475
Epoch 8/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.6500 - loss: 1.7386
Epoch 9/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.7000 - loss: 1.6142
Epoch 10/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7000 - loss: 1.4963
Epoch 11/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7000 - loss: 1.3724
Epoch 12/150
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7000

Conclusion

Successfully implemented a Sequence-to-Sequence (Seq2Seq) model using real text sequence data with Encoder-Decoder LSTM architecture in TensorFlow and Keras.